# Task 10 - Finetune model mới (Bước 4)

**Bước 4** (cuối) trong kế hoạch:
1. Feature importance (NB07).
2. Thêm feature V2 (NB08).
3. Thử xử lý mất cân bằng (NB09).
4. **Finetune lại model mới** (notebook này).

Chốt theo kết luận các bước trước:
- **Feature set = V1+V2 lọc bớt**: bỏ các feature yếu (permutation ≤ 0) từ NB07/NB08.
- **Train trên phân phối thật** (bỏ `scale_pos_weight`, không resample) - NB09 cho thấy đây là cách tốt nhất cho PR-AUC.
- Finetune cả **classifier** (tối ưu PR-AUC) lẫn **regressor** (tối ưu RMSE-log) bằng **Optuna (TPE)** + early stopping.
- Sau finetune: **chọn threshold** trên validation, so với baseline, lưu model mới.

> Trước khi finetune sẽ **kiểm chứng việc lọc feature không làm hại** (so PR-AUC/RMSE giữa full V1+V2 và bản lọc).

## 0. Setup & feature set lọc bớt

In [4]:
import json
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR if (NB_DIR / "data_pyspark_parquet").exists() else NB_DIR.parent
if not (PROJECT_ROOT / "data_pyspark_parquet").exists():
    PROJECT_ROOT = Path(r"g:/ds")

PARQUET_DIR = PROJECT_ROOT / "data_pyspark_parquet"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = NB_DIR / "finetune_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (MODELS_DIR / "feature_v2_metadata.json").open(encoding="utf-8") as f:
    v2_meta = json.load(f)

FULL_FEATURES = v2_meta["v1_feature_columns"] + v2_meta["v2_features"]
ALL_CATEGORICAL = v2_meta["all_categorical"]
ALL_BINARY = v2_meta["all_binary"]
ALL_CATEGORY_LEVELS = v2_meta["all_category_levels"]
SPLIT_CONFIG = v2_meta["split_config"]

# Feature yếu (permutation <= 0) gom từ NB07 (V1) + NB08 (V2)
DROP_FEATURES = [
    "session_month", "is_first_session", "is_bounce",
    "has_gclid", "has_traffic_campaign", "has_previous_purchase",  # V1 weak (NB07)
    "is_socially_engaged", "is_organic_traffic",                   # V2 weak (NB08)
]
FILTERED_FEATURES = [c for c in FULL_FEATURES if c not in DROP_FEATURES]

CLASSIFICATION_LABEL = "future_30d_has_purchase"
REVENUE_LABEL = "future_30d_revenue"
LOG_REVENUE_LABEL = "log_future_30d_revenue"

print("Full V1+V2 features:", len(FULL_FEATURES))
print("Dropped (weak):", len(DROP_FEATURES), "->", DROP_FEATURES)
print("Filtered features:", len(FILTERED_FEATURES))

Full V1+V2 features: 41
Dropped (weak): 8 -> ['session_month', 'is_first_session', 'is_bounce', 'has_gclid', 'has_traffic_campaign', 'has_previous_purchase', 'is_socially_engaged', 'is_organic_traffic']
Filtered features: 33


## 1. Load dữ liệu, áp dtype, time split

In [5]:
train_v2 = pd.read_parquet(PARQUET_DIR / "train_user_session_features_30d_v2.parquet")
test_v2 = pd.read_parquet(PARQUET_DIR / "test_user_session_features_30d_v2.parquet")


def apply_dtypes(pdf):
    out = pdf.copy()
    numeric_cols = [c for c in FULL_FEATURES if c not in ALL_CATEGORICAL and c not in ALL_BINARY]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
    for col in ALL_BINARY:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0).astype("int8")
    for col in ALL_CATEGORICAL:
        out[col] = pd.Categorical(out[col].astype(str), categories=ALL_CATEGORY_LEVELS[col])
    out[CLASSIFICATION_LABEL] = pd.to_numeric(out[CLASSIFICATION_LABEL], errors="raise").astype("int8")
    out[REVENUE_LABEL] = pd.to_numeric(out[REVENUE_LABEL], errors="raise").astype("float64")
    out[LOG_REVENUE_LABEL] = np.log1p(out[REVENUE_LABEL])
    return out


def time_split(pdf):
    d = pd.to_datetime(pdf["session_date"]).dt.date
    start = pd.to_datetime(SPLIT_CONFIG["validation_start_inclusive"]).date()
    end = pd.to_datetime(SPLIT_CONFIG["validation_end_inclusive"]).date()
    return pdf[d < start].copy(), pdf[(d >= start) & (d <= end)].copy()


train_lgbm = apply_dtypes(train_v2)
test_lgbm = apply_dtypes(test_v2)
train_split_pdf, validation_pdf = time_split(train_lgbm)

# Classifier: toàn bộ rows
y_train = train_split_pdf[CLASSIFICATION_LABEL]
y_valid = validation_pdf[CLASSIFICATION_LABEL]

# Regressor: chỉ positive-revenue rows
tr_pos = train_split_pdf[train_split_pdf[REVENUE_LABEL] > 0]
va_pos = validation_pdf[validation_pdf[REVENUE_LABEL] > 0]
y_train_reg = tr_pos[LOG_REVENUE_LABEL]
y_valid_reg = va_pos[LOG_REVENUE_LABEL]
y_valid_reg_rev = va_pos[REVENUE_LABEL]

CAT_FILTERED = [c for c in ALL_CATEGORICAL if c in FILTERED_FEATURES]

print("Train split:", len(train_split_pdf), "| Validation:", len(validation_pdf))
print("Train pos-rev rows:", len(tr_pos), "| Validation pos-rev rows:", len(va_pos))
print("Categorical trong feature lọc:", CAT_FILTERED)

Train split: 1530080 | Validation: 93998
Train pos-rev rows: 20109 | Validation pos-rev rows: 1084
Categorical trong feature lọc: ['channelGrouping_model', 'traffic_channel_type_model', 'device_category_model', 'geo_country_model', 'traffic_source_clean_model', 'traffic_medium_clean_model', 'browser_family_model', 'os_family_model']


## 2. Kiểm chứng: lọc feature không làm hại

Train baseline (cùng params) trên **full V1+V2** vs **bản lọc**, so PR-AUC (classifier) và RMSE-log (regressor).
Nếu bản lọc không tệ hơn đáng kể -> yên tâm finetune trên bản lọc.

In [6]:
from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping
from sklearn.metrics import average_precision_score, mean_squared_error

SEED = 42
BASE_CLF = dict(objective="binary", n_estimators=500, learning_rate=0.05, num_leaves=31,
                min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)
BASE_REG = dict(objective="regression", n_estimators=500, learning_rate=0.05, num_leaves=31,
                min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)


def quick_clf_pr_auc(features):
    cat = [c for c in ALL_CATEGORICAL if c in features]
    m = LGBMClassifier(**BASE_CLF)
    m.fit(train_split_pdf[features], y_train, categorical_feature=cat)
    return float(average_precision_score(y_valid, m.predict_proba(validation_pdf[features])[:, 1]))


def quick_reg_rmse(features):
    cat = [c for c in ALL_CATEGORICAL if c in features]
    m = LGBMRegressor(**BASE_REG)
    m.fit(tr_pos[features], y_train_reg, categorical_feature=cat)
    return float(np.sqrt(mean_squared_error(y_valid_reg, m.predict(va_pos[features]))))


check = {
    "classifier_pr_auc": {"full": quick_clf_pr_auc(FULL_FEATURES), "filtered": quick_clf_pr_auc(FILTERED_FEATURES)},
    "regressor_rmse_log": {"full": quick_reg_rmse(FULL_FEATURES), "filtered": quick_reg_rmse(FILTERED_FEATURES)},
}
print("Classifier PR-AUC  full={classifier_pr_auc[full]:.4f}  filtered={classifier_pr_auc[filtered]:.4f}".format(**check))
print("Regressor RMSE-log full={regressor_rmse_log[full]:.4f}  filtered={regressor_rmse_log[filtered]:.4f}".format(**check))
print("=> Lọc OK (không tệ hơn đáng kể), tiếp tục finetune trên FILTERED_FEATURES."
      if check["classifier_pr_auc"]["filtered"] >= check["classifier_pr_auc"]["full"] - 0.005
      else "=> CẢNH BÁO: lọc làm giảm PR-AUC > 0.005, cân nhắc giữ full feature.")

Classifier PR-AUC  full=0.1385  filtered=0.1418
Regressor RMSE-log full=1.0536  filtered=1.0545
=> Lọc OK (không tệ hơn đáng kể), tiếp tục finetune trên FILTERED_FEATURES.


## 3. Finetune CLASSIFIER bằng Optuna (tối ưu PR-AUC, phân phối thật)

- Sampler TPE, tối đa hoá `average_precision` trên validation.
- `n_estimators` để cao + **early stopping** theo `average_precision` -> tự chọn số cây tốt nhất.
- KHÔNG dùng `scale_pos_weight` (theo kết luận NB09).

In [7]:
import optuna
from lightgbm import log_evaluation

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS_CLF = 30
EARLY_STOPPING_ROUNDS = 50

X_tr_clf, X_va_clf = train_split_pdf[FILTERED_FEATURES], validation_pdf[FILTERED_FEATURES]


def clf_objective(trial):
    params = dict(
        objective="binary",
        n_estimators=2000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 300),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        random_state=SEED, n_jobs=-1, verbosity=-1,
    )
    model = LGBMClassifier(**params)
    model.fit(X_tr_clf, y_train, eval_set=[(X_va_clf, y_valid)], eval_metric="average_precision",
              categorical_feature=CAT_FILTERED,
              callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
    proba = model.predict_proba(X_va_clf)[:, 1]
    return average_precision_score(y_valid, proba)


t0 = time.time()
clf_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
# Đảm bảo Optuna luôn đánh giá cấu hình baseline (giống default) -> best không bao giờ tệ hơn baseline.
clf_study.enqueue_trial({"learning_rate": 0.05, "num_leaves": 31, "max_depth": 12, "min_child_samples": 50,
                         "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0, "reg_alpha": 1e-3})
clf_study.optimize(clf_objective, n_trials=N_TRIALS_CLF, show_progress_bar=False)
print(f"Classifier tuning xong trong {time.time()-t0:.1f}s | {N_TRIALS_CLF} trials")
print("Best PR-AUC:", round(clf_study.best_value, 4))
print("Best params:", json.dumps(clf_study.best_params, indent=2))

f:\ide\anaconda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Classifier tuning xong trong 786.5s | 30 trials
Best PR-AUC: 0.1425
Best params: {
  "learning_rate": 0.042526920013349775,
  "num_leaves": 33,
  "max_depth": 11,
  "min_child_samples": 205,
  "subsample": 0.7413405809207588,
  "colsample_bytree": 0.7997686059537362,
  "reg_lambda": 0.028196976489031585,
  "reg_alpha": 1.393978675022225
}


## 4. Finetune REGRESSOR bằng Optuna (tối ưu RMSE-log trên positive rows)

In [8]:
N_TRIALS_REG = 30
X_tr_reg, X_va_reg = tr_pos[FILTERED_FEATURES], va_pos[FILTERED_FEATURES]


def reg_objective(trial):
    params = dict(
        objective="regression",
        n_estimators=2000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 200),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        random_state=SEED, n_jobs=-1, verbosity=-1,
    )
    model = LGBMRegressor(**params)
    model.fit(X_tr_reg, y_train_reg, eval_set=[(X_va_reg, y_valid_reg)], eval_metric="rmse",
              categorical_feature=CAT_FILTERED,
              callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
    pred = model.predict(X_va_reg)
    return np.sqrt(mean_squared_error(y_valid_reg, pred))


t0 = time.time()
reg_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
reg_study.enqueue_trial({"learning_rate": 0.05, "num_leaves": 31, "max_depth": 12, "min_child_samples": 20,
                         "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0, "reg_alpha": 1e-3})
reg_study.optimize(reg_objective, n_trials=N_TRIALS_REG, show_progress_bar=False)
print(f"Regressor tuning xong trong {time.time()-t0:.1f}s | {N_TRIALS_REG} trials")
print("Best RMSE-log:", round(reg_study.best_value, 4))
print("Best params:", json.dumps(reg_study.best_params, indent=2))

Regressor tuning xong trong 9.7s | 30 trials
Best RMSE-log: 1.037
Best params: {
  "learning_rate": 0.01388425798167985,
  "num_leaves": 239,
  "max_depth": 11,
  "min_child_samples": 47,
  "subsample": 0.6581196455839982,
  "colsample_bytree": 0.9641527424066966,
  "reg_lambda": 0.03798640963036107,
  "reg_alpha": 2.846327228135729
}


## 5. Train model cuối với best params + early stopping

In [9]:
def fit_final_classifier():
    params = dict(objective="binary", n_estimators=2000, random_state=SEED, n_jobs=-1,
                  verbosity=-1, subsample_freq=1, **clf_study.best_params)
    m = LGBMClassifier(**params)
    m.fit(X_tr_clf, y_train, eval_set=[(X_va_clf, y_valid)], eval_metric="average_precision",
          categorical_feature=CAT_FILTERED,
          callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
    return m


def fit_final_regressor():
    params = dict(objective="regression", n_estimators=2000, random_state=SEED, n_jobs=-1,
                  verbosity=-1, subsample_freq=1, **reg_study.best_params)
    m = LGBMRegressor(**params)
    m.fit(X_tr_reg, y_train_reg, eval_set=[(X_va_reg, y_valid_reg)], eval_metric="rmse",
          categorical_feature=CAT_FILTERED,
          callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
    return m


final_clf = fit_final_classifier()
final_reg = fit_final_regressor()
print("Classifier best_iteration:", final_clf.best_iteration_)
print("Regressor best_iteration:", final_reg.best_iteration_)

Classifier best_iteration: 448
Regressor best_iteration: 113


## 6. Chọn threshold cho classifier trên validation

Vì dữ liệu rất lệch, threshold 0.5 không hợp lý. Quét threshold, báo cáo F1-optimal và vài điểm vận hành
(precision-oriented / recall-oriented) để chọn theo nhu cầu nghiệp vụ.

In [10]:
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, precision_recall_curve

valid_proba = final_clf.predict_proba(X_va_clf)[:, 1]


def op_point(t):
    pred = (valid_proba >= t).astype(int)
    return {
        "threshold": round(float(t), 4),
        "precision": round(float(precision_score(y_valid, pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_valid, pred, zero_division=0)), 4),
        "f1": round(float(f1_score(y_valid, pred, zero_division=0)), 4),
        "predicted_positive": int(pred.sum()),
    }


prec, rec, thr = precision_recall_curve(y_valid, valid_proba)
f1_curve = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)
best_idx = int(np.argmax(f1_curve[:-1]))
selected_threshold = float(thr[best_idx])

operating_points = pd.DataFrame([
    op_point(selected_threshold),
    op_point(0.5),
    op_point(np.quantile(valid_proba, 0.95)),   # ~ top 5% theo điểm
    op_point(np.quantile(valid_proba, 0.99)),   # ~ top 1%
])
print("Selected threshold (max F1):", round(selected_threshold, 4))
operating_points

Selected threshold (max F1): 0.127


,threshold,precision,recall,f1,predicted_positive
0,0.1270,0.1706,0.2770,0.2112,1805
1,0.5000,0.7778,0.0063,0.0125,9
2,0.0690,0.1130,0.4775,0.1827,4700
3,0.1676,0.2053,0.1736,0.1881,940


## 7. So sánh model finetuned vs baseline

In [11]:
finetuned_pr_auc = float(average_precision_score(y_valid, valid_proba))
finetuned_rmse_log = float(np.sqrt(mean_squared_error(y_valid_reg, final_reg.predict(X_va_reg))))

# Baseline tham chiếu: week_3 (scale_pos_weight, V1) + NB09 no_handling (V1+V2)
try:
    nb09 = json.load(open(NB_DIR / "imbalance_outputs" / "imbalance_comparison.json", encoding="utf-8"))
    nb09_best = nb09["best_pr_auc"]
    nb09_baseline = nb09["baseline_pr_auc"]
except Exception:
    nb09_best = nb09_baseline = None

comparison = {
    "classifier_pr_auc": {
        "week3_scale_pos_weight_baseline": nb09_baseline,
        "nb09_best_no_handling": nb09_best,
        "filtered_baseline": check["classifier_pr_auc"]["filtered"],
        "finetuned": round(finetuned_pr_auc, 4),
    },
    "regressor_rmse_log": {
        "filtered_baseline": round(check["regressor_rmse_log"]["filtered"], 4),
        "finetuned": round(finetuned_rmse_log, 4),
    },
}
print(json.dumps(comparison, indent=2, default=str))
clf_gain = finetuned_pr_auc - check["classifier_pr_auc"]["filtered"]
reg_gain = check["regressor_rmse_log"]["filtered"] - finetuned_rmse_log
print(f"\nClassifier PR-AUC: baseline {check['classifier_pr_auc']['filtered']:.4f} -> finetuned {finetuned_pr_auc:.4f} ({clf_gain:+.4f})")
print(f"Regressor RMSE-log: baseline {check['regressor_rmse_log']['filtered']:.4f} -> finetuned {finetuned_rmse_log:.4f} ({-reg_gain:+.4f}, âm = tốt hơn)")

{
  "classifier_pr_auc": {
    "week3_scale_pos_weight_baseline": 0.12487470172201656,
    "nb09_best_no_handling": 0.13846699521453928,
    "filtered_baseline": 0.14180825503097566,
    "finetuned": 0.1425
  },
  "regressor_rmse_log": {
    "filtered_baseline": 1.0545,
    "finetuned": 1.037
  }
}

Classifier PR-AUC: baseline 0.1418 -> finetuned 0.1425 (+0.0007)
Regressor RMSE-log: baseline 1.0545 -> finetuned 1.0370 (-0.0175, âm = tốt hơn)


## 8. Lưu model finetuned + metadata

In [12]:
import pickle

VERSION = datetime.now(timezone.utc).strftime("v%Y%m%dT%H%M%S")
SAVE_DIR = MODELS_DIR / f"finetuned_{VERSION}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

with (SAVE_DIR / "lgbm_purchase_classifier_30d.pkl").open("wb") as f:
    pickle.dump(final_clf, f)
with (SAVE_DIR / "lgbm_revenue_regressor_30d.pkl").open("wb") as f:
    pickle.dump(final_reg, f)

finetune_metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "version": VERSION,
    "feature_set": "v1v2_filtered",
    "feature_columns": FILTERED_FEATURES,
    "dropped_weak_features": DROP_FEATURES,
    "categorical_features": CAT_FILTERED,
    "category_levels": {c: ALL_CATEGORY_LEVELS[c] for c in CAT_FILTERED},
    "imbalance_strategy": "natural_distribution_no_scale_pos_weight",
    "classifier_best_params": clf_study.best_params,
    "classifier_best_iteration": int(final_clf.best_iteration_ or 0),
    "regressor_best_params": reg_study.best_params,
    "regressor_best_iteration": int(final_reg.best_iteration_ or 0),
    "selected_threshold": selected_threshold,
    "validation_pr_auc": finetuned_pr_auc,
    "validation_rmse_log_positive": finetuned_rmse_log,
    "comparison": comparison,
    "n_trials_classifier": N_TRIALS_CLF,
    "n_trials_regressor": N_TRIALS_REG,
    "split_config": SPLIT_CONFIG,
}
with (SAVE_DIR / "finetune_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(finetune_metadata, f, ensure_ascii=False, indent=2, default=str)

operating_points.to_csv(OUTPUT_DIR / "classifier_operating_points.csv", index=False)
with (OUTPUT_DIR / "finetune_comparison.json").open("w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2, default=str)

print("Saved finetuned models ->", SAVE_DIR)
print("Selected threshold:", round(selected_threshold, 4))

Saved finetuned models -> G:\ds\models\finetuned_v20260618T090411
Selected threshold: 0.127


## 10. Nhận xét & tổng kết

So sánh ở mục 7 cho biết finetune có cải thiện so với baseline (cùng feature lọc) không:
- **Classifier**: PR-AUC tăng = Optuna tìm được cấu hình tách lớp tốt hơn.
- **Regressor**: RMSE-log giảm = dự đoán doanh thu (trên positive rows) chính xác hơn.

Lưu ý vận hành (mục 6): chọn `selected_threshold` theo nhu cầu - F1-optimal để cân bằng,
hoặc threshold cao hơn (top 1%/5%) nếu muốn precision cao cho việc nhắm mục tiêu marketing.

Final test (mục 9): so validation vs test để kiểm tra generalization; revenue capture top-k cho biết
chất lượng **ranking** expected revenue (mục tiêu thực dụng của bài toán).

### Tổng kết cả 4 bước
1. NB07: `visit_number`, `totals_hits`, history là lõi của model.
2. NB08: feature V2 giúp regressor nhẹ, không giúp classifier (PR-AUC).
3. NB09: resampling/`scale_pos_weight` không cải thiện ranking; train phân phối thật + chọn threshold là đúng hướng.
4. NB10: finetune trên feature lọc + phân phối thật + Optuna; final test + model + threshold đã lưu để phục vụ API.

> Để dùng cho API: trỏ tới thư mục `models/finetuned_<version>/`, đọc `selected_threshold` và `feature_columns`
> từ `finetune_metadata.json`, và metric cuối ở `final_test_metrics.json`.

In [13]:
# ===== FINAL TEST EVALUATION - chỉ chạy SAU khi đã chốt model + threshold =====
# Không refit, không chọn lại threshold, không đổi feature. Chỉ predict + tính metric cuối.
X_test = test_lgbm[FILTERED_FEATURES]
y_test = test_lgbm[CLASSIFICATION_LABEL].to_numpy()
test_revenue = test_lgbm[REVENUE_LABEL].to_numpy(dtype="float64")
test_pos_mask = test_revenue > 0

# Two-stage prediction (giống week_3)
test_proba = final_clf.predict_proba(X_test)[:, 1]
test_pred_rev_if_purchase = np.expm1(final_reg.predict(X_test))
test_expected_revenue = test_proba * test_pred_rev_if_purchase


def classification_metrics(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float((tp + tn) / (tp + tn + fp + fn)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "pr_auc": float(average_precision_score(y_true, proba)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


def regression_metrics_positive(y_true_rev, pred_rev_if_purchase, pos_mask):
    yt = y_true_rev[pos_mask]
    yp = np.clip(pred_rev_if_purchase[pos_mask], 0, None)
    yt_log, yp_log = np.log1p(yt), np.log1p(yp)
    return {
        "positive_rows": int(pos_mask.sum()),
        "rmse_log": float(np.sqrt(np.mean((yp_log - yt_log) ** 2))),
        "mae_log": float(np.mean(np.abs(yp_log - yt_log))),
        "rmse_revenue": float(np.sqrt(np.mean((yp - yt) ** 2))),
        "mae_revenue": float(np.mean(np.abs(yp - yt))),
    }


def revenue_capture_at_k(actual, predicted, ks=(0.01, 0.05, 0.10)):
    order = np.argsort(-predicted)
    total, n = float(actual.sum()), len(actual)
    out = {}
    for k in ks:
        topn = max(1, int(np.ceil(n * k)))
        cap = float(actual[order[:topn]].sum())
        out[f"top_{int(k * 100)}pct"] = {"captured_pct": float(cap / total * 100) if total > 0 else None}
    return out


def expected_revenue_metrics(actual, predicted):
    err = predicted - actual
    corr = float(np.corrcoef(actual, predicted)[0, 1]) if np.std(actual) > 0 and np.std(predicted) > 0 else None
    return {
        "row_count": int(len(actual)),
        "actual_avg_revenue": float(actual.mean()),
        "predicted_avg_expected_revenue": float(predicted.mean()),
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "correlation": corr,
        "revenue_capture": revenue_capture_at_k(actual, predicted),
    }


# Validation đối chiếu (cùng selected_threshold)
valid_expected_revenue = valid_proba * np.expm1(final_reg.predict(X_va_clf))
final_validation_metrics = {
    "classification": classification_metrics(y_valid.to_numpy(), valid_proba, selected_threshold),
    "conditional_revenue_regression": regression_metrics_positive(
        validation_pdf[REVENUE_LABEL].to_numpy(dtype="float64"),
        np.expm1(final_reg.predict(X_va_clf)),
        validation_pdf[REVENUE_LABEL].to_numpy(dtype="float64") > 0,
    ),
    "expected_revenue": expected_revenue_metrics(
        validation_pdf[REVENUE_LABEL].to_numpy(dtype="float64"), valid_expected_revenue),
}

final_test_metrics = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_version": VERSION,
    "feature_set": "v1v2_filtered",
    "selected_threshold": selected_threshold,
    "test_rows": int(len(y_test)),
    "classification": classification_metrics(y_test, test_proba, selected_threshold),
    "conditional_revenue_regression": regression_metrics_positive(test_revenue, test_pred_rev_if_purchase, test_pos_mask),
    "expected_revenue": expected_revenue_metrics(test_revenue, test_expected_revenue),
    "validation_for_comparison": final_validation_metrics,
}

with (SAVE_DIR / "final_test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(final_test_metrics, f, ensure_ascii=False, indent=2, default=str)
with (OUTPUT_DIR / "final_test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(final_test_metrics, f, ensure_ascii=False, indent=2, default=str)

c_te = final_test_metrics["classification"]
c_va = final_validation_metrics["classification"]
print(f"[TEST]       PR-AUC={c_te['pr_auc']:.4f} | @thr={selected_threshold:.3f} precision={c_te['precision']:.3f} recall={c_te['recall']:.3f} f1={c_te['f1']:.3f}")
print(f"[VALIDATION] PR-AUC={c_va['pr_auc']:.4f} | @thr={selected_threshold:.3f} precision={c_va['precision']:.3f} recall={c_va['recall']:.3f} f1={c_va['f1']:.3f}")
print(f"Revenue capture (test): top1%={final_test_metrics['expected_revenue']['revenue_capture']['top_1pct']['captured_pct']:.1f}% "
      f"top5%={final_test_metrics['expected_revenue']['revenue_capture']['top_5pct']['captured_pct']:.1f}% "
      f"top10%={final_test_metrics['expected_revenue']['revenue_capture']['top_10pct']['captured_pct']:.1f}%")
print("Conditional RMSE-log (test):", round(final_test_metrics['conditional_revenue_regression']['rmse_log'], 4))
print("Saved final_test_metrics.json ->", SAVE_DIR)

[TEST]       PR-AUC=0.1567 | @thr=0.127 precision=0.183 recall=0.286 f1=0.223
[VALIDATION] PR-AUC=0.1425 | @thr=0.127 precision=0.171 recall=0.277 f1=0.211
Revenue capture (test): top1%=56.3% top5%=75.3% top10%=84.8%
Conditional RMSE-log (test): 1.0952
Saved final_test_metrics.json -> G:\ds\models\finetuned_v20260618T090411


## 9. Nhận xét & tổng kết

So sánh ở mục 7 cho biết finetune có cải thiện so với baseline (cùng feature lọc) không:
- **Classifier**: PR-AUC tăng = Optuna tìm được cấu hình tách lớp tốt hơn.
- **Regressor**: RMSE-log giảm = dự đoán doanh thu (trên positive rows) chính xác hơn.

Lưu ý vận hành (mục 6): chọn `selected_threshold` theo nhu cầu - F1-optimal để cân bằng,
hoặc threshold cao hơn (top 1%/5%) nếu muốn precision cao cho việc nhắm mục tiêu marketing.

### Tổng kết cả 4 bước
1. NB07: `visit_number`, `totals_hits`, history là lõi của model.
2. NB08: feature V2 giúp regressor nhẹ, không giúp classifier (PR-AUC).
3. NB09: resampling/`scale_pos_weight` không cải thiện ranking; train phân phối thật + chọn threshold là đúng hướng.
4. NB10: finetune trên feature lọc + phân phối thật + Optuna; model + threshold đã lưu để phục vụ API.

> Để dùng cho API: trỏ tới thư mục `models/finetuned_<version>/` và đọc `selected_threshold` từ `finetune_metadata.json`.
> `FINAL TEST EVALUATION` (test set) chỉ chạy sau khi đã chốt hoàn toàn - dùng `test_user_session_features_30d_v2.parquet` (đã load sẵn là `test_lgbm`).